# Task 2: Insurance Data Cleaning (Premium Prediction)

Clean the raw insurance dataset for predicting 2021 insurance premiums using 2018-2020 data.

**Target**: `Earned Premium` (continuous — regression problem)

**Steps:**
1. Load & explore
2. Drop structurally absent columns (fire-event-specific fields)
3. Drop claims columns (prevent data leakage)
4. Handle missing values (including rescuing Avg PPC)
5. Save cleaned data with Year column for temporal train/test split

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/insurance_data.csv", low_memory=False)
print("Shape:", df.shape)
print(f"Years: {sorted(df['Year'].unique())}")
print(f"Rows per year:")
print(df['Year'].value_counts().sort_index())
df.head()

Shape: (47033, 76)
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Rows per year:
Year
2018    10071
2019     9818
2020    13800
2021    13344
Name: count, dtype: int64


,Year,ZIP,Avg Fire Risk Score,Avg PPC,CAT Cov A Fire - Incurred Losses,CAT Cov A Fire - Number of Claims,CAT Cov A Smoke - Incurred Losses,CAT Cov A Smoke - Number of Claims,CAT Cov C Fire - Incurred Losses,CAT Cov C Fire - Number of Claims,...,average_household_size,educational_attainment_bachelor_or_higher,poverty_status,housing_occupancy_number,housing_value,year_structure_built,housing_vacancy_number,median_monthly_housing_costs,owner_occupied_housing_units,renter_occupied_housing_units
0,2018,90003,1.000,1.000,0,0,0,0,0,0,...,4.03,2294.0,18743.0,18349.0,547600.0,18349.0,715.0,1609.0,4692.0,12942.0
1,2018,90004,0.420,1.075,0,0,0,0,0,0,...,2.44,11822.0,10994.0,26046.0,1457200.0,26046.0,2311.0,1847.0,4011.0,19724.0
2,2018,90005,0.510,1.060,0,0,0,0,0,0,...,2.18,7582.0,9463.0,19357.0,1084400.0,19357.0,1880.0,1651.0,1572.0,15905.0
3,2018,90006,0.605,1.040,0,0,0,0,0,0,...,2.79,6758.0,13274.0,21475.0,841900.0,21475.0,2207.0,1415.0,1951.0,17317.0
4,2018,90007,0.000,2.000,0,0,0,0,0,0,...,2.75,3536.0,13473.0,14153.0,852900.0,14153.0,1702.0,1504.0,1297.0,11154.0


In [2]:
# Check missing % per column
missing_pct = df.isna().mean().sort_values(ascending=False)
print("Missing % per column:\n")
print(missing_pct.to_string())

Missing % per column:

FIRE_NUM                                     1.000000
COMPLEX_ID                                   0.998724
COMPLEX_NA                                   0.997704
COMMENTS                                     0.956754
INC_NUM                                      0.834031
CONT_DATE                                    0.832692
UNIT_ID                                      0.832118
FIRE_NAME                                    0.831756
station                                      0.831714
avg_tmax_c                                   0.831714
avg_tmin_c                                   0.831714
tot_prcp_mm                                  0.831714
Alarm_Date2                                  0.831374
year_month                                   0.831374
ALARM_DATE                                   0.831374
C_METHOD                                     0.831267
AGENCY_ID                                    0.831012
GIS_ACRES                                    0.831012
CAUSE

## Step 2: Drop Structurally Absent Columns

These columns are only populated when a fire event occurred (~83% missing). They represent fire incident data (names, dates, causes, GIS coordinates, weather during fires) — NOT predictive features. At prediction time, we wouldn't have this data.

In [3]:
# Drop fire-event-specific columns (structurally absent — only exist when a fire occurred)
fire_event_cols = [
    # Fire metadata
    "FIRE_NUM", "FIRE_NAME", "FIRE_NAME_ID", "COMPLEX_ID", "COMPLEX_NA",
    "INC_NUM", "AGENCY", "AGENCY_ID", "UNIT_ID", "OBJECTID",
    "CAUSE", "C_METHOD", "OBJECTIVE", "COMMENTS", "DECADES",
    # Fire dates
    "ALARM_DATE", "Alarm_Date2", "CONT_DATE", "year_month",
    # Fire GIS / location
    "latitude", "longitude", "Shape__Len", "Shape__Are", "GIS_ACRES",
    # Fire weather (conditions DURING fire, not general climate)
    "station", "avg_tmax_c", "avg_tmin_c", "tot_prcp_mm",
]

# Only drop columns that actually exist in the dataframe
cols_to_drop = [c for c in fire_event_cols if c in df.columns]
print(f"Dropping {len(cols_to_drop)} fire-event columns:")
for c in cols_to_drop:
    print(f"  {c}: {df[c].isna().mean():.1%} missing")

df = df.drop(columns=cols_to_drop)
print(f"\nRemaining shape: {df.shape}")

Dropping 28 fire-event columns:
  FIRE_NUM: 100.0% missing
  FIRE_NAME: 83.2% missing
  FIRE_NAME_ID: 83.1% missing
  COMPLEX_ID: 99.9% missing
  COMPLEX_NA: 99.8% missing
  INC_NUM: 83.4% missing
  AGENCY: 83.1% missing
  AGENCY_ID: 83.1% missing
  UNIT_ID: 83.2% missing
  OBJECTID: 83.1% missing
  CAUSE: 83.1% missing
  C_METHOD: 83.1% missing
  OBJECTIVE: 83.1% missing
  COMMENTS: 95.7% missing
  DECADES: 83.1% missing
  ALARM_DATE: 83.1% missing
  Alarm_Date2: 83.1% missing
  CONT_DATE: 83.3% missing
  year_month: 83.1% missing
  latitude: 83.1% missing
  longitude: 83.1% missing
  Shape__Len: 83.1% missing
  Shape__Are: 83.1% missing
  GIS_ACRES: 83.1% missing
  station: 83.2% missing
  avg_tmax_c: 83.2% missing
  avg_tmin_c: 83.2% missing
  tot_prcp_mm: 83.2% missing

Remaining shape: (47033, 48)


## Step 3: Drop Claims Columns (Prevent Data Leakage)

Claims (fire & smoke, incurred losses & number of claims) are **outcomes**, not predictors. You can't know how many claims will be filed when setting premiums. Keeping these would give the model the answer.

In [4]:
# Drop all claims columns (these are outcomes, not predictors — data leakage)
claims_cols = [c for c in df.columns if "Incurred Losses" in c or "Number of Claims" in c]

print(f"Dropping {len(claims_cols)} claims columns:")
for c in claims_cols:
    print(f"  {c}")

df = df.drop(columns=claims_cols)
print(f"\nRemaining shape: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")

Dropping 16 claims columns:
  CAT Cov A Fire -  Incurred Losses
  CAT Cov A Fire -  Number of Claims
  CAT Cov A Smoke -  Incurred Losses
  CAT Cov A Smoke -  Number of Claims
  CAT Cov C Fire -  Incurred Losses
  CAT Cov C Fire -  Number of Claims
  CAT Cov C Smoke -  Incurred Losses
  CAT Cov C Smoke -  Number of Claims
  Non-CAT Cov A Fire -  Incurred Losses
  Non-CAT Cov A Fire -  Number of Claims
  Non-CAT Cov A Smoke -  Incurred Losses
  Non-CAT Cov A Smoke -  Number of Claims
  Non-CAT Cov C Fire -  Incurred Losses
  Non-CAT Cov C Fire -  Number of Claims
  Non-CAT Cov C Smoke -  Incurred Losses
  Non-CAT Cov C Smoke -  Number of Claims

Remaining shape: (47033, 32)
Remaining columns: ['Year', 'ZIP', 'Avg Fire Risk Score', 'Avg PPC', 'Cov A Amount Weighted Avg', 'Cov C Amount Weighted Avg', 'Earned Exposure', 'Earned Premium', 'Number of High Fire Risk Exposure', 'Number of Low Fire Risk Exposure', 'Number of Moderate Fire Risk Exposure', 'Number of Negligible Fire Risk Exposure

## Step 4: Verify Target Variable

`Earned Premium` is our target — the premium revenue we're predicting for 2021.

In [5]:
# Verify target variable
print("Target: Earned Premium")
print(f"Missing: {df['Earned Premium'].isna().sum()}")
print(f"\nDistribution:")
print(df['Earned Premium'].describe())
print(f"\nBy year:")
print(df.groupby('Year')['Earned Premium'].describe())

Target: Earned Premium
Missing: 0

Distribution:
count    4.703300e+04
mean     5.079100e+05
std      1.455284e+06
min     -5.590000e+02
25%      2.842000e+03
50%      4.420700e+04
75%      2.648910e+05
max      3.627053e+07
Name: Earned Premium, dtype: float64

By year:
        count           mean           std    min     25%      50%        75%  \
Year                                                                            
2018  10071.0  423751.610168  1.152239e+06 -559.0  6969.0  41366.0  217059.50   
2019   9818.0  452804.680892  1.210065e+06 -496.0  7706.0  46523.0  237814.25   
2020  13800.0  526657.010652  1.533476e+06   -3.0   540.0  41829.5  277872.75   
2021  13344.0  592582.911871  1.714285e+06   -1.0   484.5  47140.0  319846.25   

             max  
Year              
2018  24942324.0  
2019  27353182.0  
2020  33151745.0  
2021  36270532.0  


## Step 5: Handle Missing Values

- `Avg PPC` (57.7% missing): Public Protection Classification — rates local fire dept quality (1-10). Directly relevant to premiums. Fill with median rather than dropping.
- Census/demographic columns (10-17% missing): Fill with median.

In [6]:
# Check remaining missing values before filling
remaining_missing = df.isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)
print(f"Columns with missing values: {len(remaining_missing)}")
print(remaining_missing)
print(f"\nAs percentages:")
print((remaining_missing / len(df) * 100).round(1))

Columns with missing values: 14
Avg PPC                                      27144
median_income                                 8217
housing_value                                 8103
median_monthly_housing_costs                  7829
average_household_size                        5902
total_population                              4654
total_housing_units                           4654
educational_attainment_bachelor_or_higher     4654
poverty_status                                4654
housing_occupancy_number                      4654
year_structure_built                          4654
housing_vacancy_number                        4654
owner_occupied_housing_units                  4654
renter_occupied_housing_units                 4654
dtype: int64

As percentages:
Avg PPC                                      57.7
median_income                                17.5
housing_value                                17.2
median_monthly_housing_costs                 16.6
average_household_size  

In [7]:
# Fill missing values with median (numeric columns)
for col in df.columns:
    if df[col].isna().sum() == 0:
        continue
    if df[col].dtype in ["float64", "int64"]:
        median_val = df[col].median()
        print(f"  Filling {col} ({df[col].isna().sum()} missing) with median: {median_val}")
        df[col] = df[col].fillna(median_val)
    else:
        mode_val = df[col].mode()[0]
        print(f"  Filling {col} ({df[col].isna().sum()} missing) with mode: {mode_val}")
        df[col] = df[col].fillna(mode_val)

# Verify no missing values remain
assert df.isna().sum().sum() == 0, "Still have missing values!"
print(f"\nAll missing values handled. Zero NaN remaining.")
print(f"Shape: {df.shape}")

  Filling Avg PPC (27144 missing) with median: 3.0
  Filling total_population (4654 missing) with median: 18255.0
  Filling median_income (8217 missing) with median: 89609.0
  Filling total_housing_units (4654 missing) with median: 7050.0
  Filling average_household_size (5902 missing) with median: 2.7
  Filling educational_attainment_bachelor_or_higher (4654 missing) with median: 2301.0
  Filling poverty_status (4654 missing) with median: 1572.0
  Filling housing_occupancy_number (4654 missing) with median: 7050.0
  Filling housing_value (8103 missing) with median: 623500.0
  Filling year_structure_built (4654 missing) with median: 7050.0
  Filling housing_vacancy_number (4654 missing) with median: 436.0
  Filling median_monthly_housing_costs (7829 missing) with median: 1884.0
  Filling owner_occupied_housing_units (4654 missing) with median: 3471.0
  Filling renter_occupied_housing_units (4654 missing) with median: 2068.0

All missing values handled. Zero NaN remaining.
Shape: (47033

## Step 6: Save Cleaned Data

Keep all 47,033 rows (no undersampling — this is regression, not classification).  
Year column is retained for train/test splitting in the modeling notebook (train: 2018-2020, test: 2021).

In [8]:
# Save cleaned dataset
df.to_csv("../data/cleaned/insurance_cleaned.csv", index=False)

print(f"Saved cleaned data to data/cleaned/insurance_cleaned.csv")
print(f"Final shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
for c in df.columns:
    print(f"  {c} ({df[c].dtype})")
print(f"\nTarget (Earned Premium) stats:")
print(df['Earned Premium'].describe())
print(f"\nRows per year:")
print(df['Year'].value_counts().sort_index())
print(f"\nMissing values: {df.isna().sum().sum()}")

Saved cleaned data to data/cleaned/insurance_cleaned.csv
Final shape: (47033, 32)

Columns (32):
  Year (int64)
  ZIP (int64)
  Avg Fire Risk Score (float64)
  Avg PPC (float64)
  Cov A Amount Weighted Avg (float64)
  Cov C Amount Weighted Avg (float64)
  Earned Exposure (float64)
  Earned Premium (int64)
  Number of High Fire Risk Exposure (int64)
  Number of Low Fire Risk Exposure (int64)
  Number of Moderate Fire Risk Exposure (int64)
  Number of Negligible Fire Risk Exposure (int64)
  Number of Very High Fire Risk Exposure (int64)
  Category_CO (bool)
  Category_DO (bool)
  Category_DT (bool)
  Category_HO (bool)
  Category_MH (bool)
  Category_RT (bool)
  total_population (float64)
  median_income (float64)
  total_housing_units (float64)
  average_household_size (float64)
  educational_attainment_bachelor_or_higher (float64)
  poverty_status (float64)
  housing_occupancy_number (float64)
  housing_value (float64)
  year_structure_built (float64)
  housing_vacancy_number (float64)